# Particion Temporal y Normalizacion con metodo por Transecto y metodo General

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 5 (adaptado a transectos): Partición temporal y normalización.
Entrada: ventanas generadas en Código 4 (carpeta windows/)
Salida: conjuntos particionados y normalizados en windows_partitioned/
Estructura:
    windows_partitioned/
        by_transect/
            ml/
                Transecto_1/
                    train_X.npy, train_y.npy, val_X.npy, val_y.npy, test_X.npy, test_y.npy
                    features.json (opcional)
                Transecto_2/ ...
            dl/
                Transecto_1/
                    ... (datos normalizados) + scaler_X.pkl, scaler_y.pkl
        global/
            ml/
                Estacion_1/ ...
            dl/
                Estacion_1/ ...
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("/usu/snsaetor/Documents/GitHub/TFGFinal/Datos_TFG_outliers/")
WINDOWS_DIR = os.path.join(BASE_DIR, "windows")
OUTPUT_DIR = os.path.join(BASE_DIR, "windows_partitioned")

# Subcarpetas de entrada (donde están los .npy)
INPUT_ML_TRANSECT = os.path.join(WINDOWS_DIR, "by_transect", "ml")
INPUT_DL_TRANSECT = os.path.join(WINDOWS_DIR, "by_transect", "dl")
INPUT_ML_GLOBAL = os.path.join(WINDOWS_DIR, "global", "ml")
INPUT_DL_GLOBAL = os.path.join(WINDOWS_DIR, "global", "dl")

# Subcarpetas de salida
OUTPUT_ML_TRANSECT = os.path.join(OUTPUT_DIR, "by_transect", "ml")
OUTPUT_DL_TRANSECT = os.path.join(OUTPUT_DIR, "by_transect", "dl")
OUTPUT_ML_GLOBAL = os.path.join(OUTPUT_DIR, "global", "ml")
OUTPUT_DL_GLOBAL = os.path.join(OUTPUT_DIR, "global", "dl")

for path in [OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL]:
    os.makedirs(path, exist_ok=True)

# Parámetros
WINDOW_IN = 72
WINDOW_OUT = 72
TARGET_COL = 'O3'

# Fechas de corte
TRAIN_END = '2025-06-30 23:00:00'
VAL_START = '2025-07-01 00:00:00'
VAL_END = '2025-10-01 23:00:00'
TEST_START = '2025-10-02 00:00:00'

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def load_entity_data(entity_dir, entity_name):
    """
    Carga los arrays X, y y timestamps para una entidad (transecto o estación)
    desde la carpeta entity_dir, donde se esperan archivos:
        {entity_name}_X.npy
        {entity_name}_y.npy
        {entity_name}_timestamps.npy
    Retorna (X, y, timestamps) o (None, None, None) si falta algún archivo.
    """
    X_path = os.path.join(entity_dir, f"{entity_name}_X.npy")
    y_path = os.path.join(entity_dir, f"{entity_name}_y.npy")
    ts_path = os.path.join(entity_dir, f"{entity_name}_timestamps.npy")
    if not (os.path.exists(X_path) and os.path.exists(y_path) and os.path.exists(ts_path)):
        print(f"    Faltan archivos para {entity_name} en {entity_dir}")
        return None, None, None
    X = np.load(X_path)
    y = np.load(y_path)
    timestamps = np.load(ts_path)
    # Convertir timestamps a datetime de pandas para facilitar comparación
    timestamps = pd.to_datetime(timestamps)
    return X, y, timestamps

def split_by_timestamps(X, y, timestamps):
    """
    Divide las ventanas según las fechas de los timestamps.
    Retorna:
        (X_train, y_train), (X_val, y_val), (X_test, y_test)
    """
    train_mask = timestamps <= pd.to_datetime(TRAIN_END)
    val_mask = (timestamps >= pd.to_datetime(VAL_START)) & (timestamps <= pd.to_datetime(VAL_END))
    test_mask = timestamps >= pd.to_datetime(TEST_START)

    # Verificar que no haya solapamiento
    assert not (train_mask & val_mask).any(), "Solapamiento train/val"
    assert not (train_mask & test_mask).any(), "Solapamiento train/test"
    assert not (val_mask & test_mask).any(), "Solapamiento val/test"

    X_train = X[train_mask]
    y_train = y[train_mask]
    X_val = X[val_mask]
    y_val = y[val_mask]
    X_test = X[test_mask]
    y_test = y[test_mask]

    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

def normalize_data(X_train, X_val, X_test, y_train, y_val, y_test):
    """
    Aplica MinMaxScaler a X e y. X tiene forma (n, window_in, n_features).
    Retorna los conjuntos escalados y los scalers.
    """
    n_train, win_in, n_feat = X_train.shape
    n_val = X_val.shape[0]
    n_test = X_test.shape[0]

    # Aplanar X para escalar
    X_train_flat = X_train.reshape(-1, n_feat)
    X_val_flat = X_val.reshape(-1, n_feat)
    X_test_flat = X_test.reshape(-1, n_feat)

    scaler_X = MinMaxScaler()
    X_train_scaled_flat = scaler_X.fit_transform(X_train_flat)
    X_val_scaled_flat = scaler_X.transform(X_val_flat)
    X_test_scaled_flat = scaler_X.transform(X_test_flat)

    # Reconstruir forma 3D
    X_train_sc = X_train_scaled_flat.reshape(n_train, win_in, n_feat)
    X_val_sc = X_val_scaled_flat.reshape(n_val, win_in, n_feat)
    X_test_sc = X_test_scaled_flat.reshape(n_test, win_in, n_feat)

    # Escalar y (target)
    scaler_y = MinMaxScaler()
    y_train_flat = y_train.reshape(-1, 1)
    y_val_flat = y_val.reshape(-1, 1)
    y_test_flat = y_test.reshape(-1, 1)

    scaler_y.fit(y_train_flat)
    y_train_scaled_flat = scaler_y.transform(y_train_flat)
    y_val_scaled_flat = scaler_y.transform(y_val_flat)
    y_test_scaled_flat = scaler_y.transform(y_test_flat)

    # CORRECCIÓN: usar WINDOW_OUT en lugar de win_out
    y_train_sc = y_train_scaled_flat.reshape(n_train, WINDOW_OUT)
    y_val_sc = y_val_scaled_flat.reshape(n_val, WINDOW_OUT)
    y_test_sc = y_test_scaled_flat.reshape(n_test, WINDOW_OUT)

    return X_train_sc, X_val_sc, X_test_sc, y_train_sc, y_val_sc, y_test_sc, scaler_X, scaler_y

def save_split_ml(output_dir, entity_name, X_train, y_train, X_val, y_val, X_test, y_test):
    """Guarda los conjuntos ML (sin escalar) en subcarpeta entity_name."""
    save_dir = os.path.join(output_dir, entity_name)
    os.makedirs(save_dir, exist_ok=True)
    np.save(os.path.join(save_dir, "train_X.npy"), X_train)
    np.save(os.path.join(save_dir, "train_y.npy"), y_train)
    np.save(os.path.join(save_dir, "val_X.npy"), X_val)
    np.save(os.path.join(save_dir, "val_y.npy"), y_val)
    np.save(os.path.join(save_dir, "test_X.npy"), X_test)
    np.save(os.path.join(save_dir, "test_y.npy"), y_test)
    # No guardamos scalers aquí

def save_split_dl(output_dir, entity_name, X_train, y_train, X_val, y_val, X_test, y_test,
                  scaler_X, scaler_y):
    """Guarda los conjuntos DL (escalados) y los scalers."""
    save_dir = os.path.join(output_dir, entity_name)
    os.makedirs(save_dir, exist_ok=True)
    np.save(os.path.join(save_dir, "train_X.npy"), X_train)
    np.save(os.path.join(save_dir, "train_y.npy"), y_train)
    np.save(os.path.join(save_dir, "val_X.npy"), X_val)
    np.save(os.path.join(save_dir, "val_y.npy"), y_val)
    np.save(os.path.join(save_dir, "test_X.npy"), X_test)
    np.save(os.path.join(save_dir, "test_y.npy"), y_test)
    with open(os.path.join(save_dir, "scaler_X.pkl"), 'wb') as f:
        pickle.dump(scaler_X, f)
    with open(os.path.join(save_dir, "scaler_y.pkl"), 'wb') as f:
        pickle.dump(scaler_y, f)

def process_entity(entity_name, input_ml_dir, input_dl_dir, output_ml_dir, output_dl_dir):
    """
    Procesa una entidad (transecto o estación) cargando datos ML y DL,
    realizando partición y, para DL, normalización.
    """
    print(f"  Procesando {entity_name}...")
    
    # Cargar datos ML
    X_ml, y_ml, ts_ml = load_entity_data(input_ml_dir, entity_name)
    if X_ml is None:
        return
    
    # Cargar datos DL (necesarios para normalizar, aunque usaremos los mismos timestamps)
    X_dl, y_dl, ts_dl = load_entity_data(input_dl_dir, entity_name)
    if X_dl is None:
        return
    
    # Verificar que timestamps coinciden
    if not np.array_equal(ts_ml, ts_dl):
        print(f"    Error: los timestamps de ML y DL no coinciden para {entity_name}. Se omite.")
        return
    
    # Partición temporal (usamos timestamps de cualquiera, son iguales)
    (X_train_ml, y_train_ml), (X_val_ml, y_val_ml), (X_test_ml, y_test_ml) = split_by_timestamps(X_ml, y_ml, ts_ml)
    (X_train_dl, y_train_dl), (X_val_dl, y_val_dl), (X_test_dl, y_test_dl) = split_by_timestamps(X_dl, y_dl, ts_dl)
    
    # Guardar versión ML (sin escalar)
    save_split_ml(output_ml_dir, entity_name,
                  X_train_ml, y_train_ml,
                  X_val_ml, y_val_ml,
                  X_test_ml, y_test_ml)
    
    # Normalizar y guardar versión DL
    if len(X_train_dl) > 0:
        X_train_sc, X_val_sc, X_test_sc, y_train_sc, y_val_sc, y_test_sc, scaler_X, scaler_y = normalize_data(
            X_train_dl, X_val_dl, X_test_dl, y_train_dl, y_val_dl, y_test_dl
        )
        save_split_dl(output_dl_dir, entity_name,
                      X_train_sc, y_train_sc,
                      X_val_sc, y_val_sc,
                      X_test_sc, y_test_sc,
                      scaler_X, scaler_y)
    else:
        print(f"    No hay datos de entrenamiento para {entity_name}, se omite DL.")

# ============================================================================
# PROCESAMIENTO POR TRANSECTO
# ============================================================================

def process_by_transect():
    print("\n--- Procesando datos por transecto ---")
    
    if not os.path.exists(INPUT_ML_TRANSECT) or not os.path.exists(INPUT_DL_TRANSECT):
        print("  Las carpetas de entrada por transecto no existen.")
        return
    
    # Obtener lista de nombres de entidades (archivos *_X.npy en ML)
    ml_files = [f.stem.replace("_X", "") for f in Path(INPUT_ML_TRANSECT).glob("*_X.npy")]
    dl_files = [f.stem.replace("_X", "") for f in Path(INPUT_DL_TRANSECT).glob("*_X.npy")]
    common = set(ml_files) & set(dl_files)
    
    if not common:
        print("  No hay entidades comunes entre ML y DL.")
        return
    
    for entity in sorted(common):
        process_entity(entity, INPUT_ML_TRANSECT, INPUT_DL_TRANSECT,
                       OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT)

# ============================================================================
# PROCESAMIENTO GLOBAL (por estación)
# ============================================================================

def process_global():
    print("\n--- Procesando datos globales (por estación) ---")
    
    if not os.path.exists(INPUT_ML_GLOBAL) or not os.path.exists(INPUT_DL_GLOBAL):
        print("  Las carpetas de entrada global no existen.")
        return
    
    ml_files = [f.stem.replace("_X", "") for f in Path(INPUT_ML_GLOBAL).glob("*_X.npy")]
    dl_files = [f.stem.replace("_X", "") for f in Path(INPUT_DL_GLOBAL).glob("*_X.npy")]
    common = set(ml_files) & set(dl_files)
    
    if not common:
        print("  No hay entidades comunes entre ML y DL global.")
        return
    
    for entity in sorted(common):
        process_entity(entity, INPUT_ML_GLOBAL, INPUT_DL_GLOBAL,
                       OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL)

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("=" * 60)
    print("Partición temporal y normalización de ventanas")
    print("=" * 60)
    
    process_by_transect()
    process_global()
    
    print("\nProceso completado. Revise la carpeta:")
    print(f"  {OUTPUT_DIR}")

Partición temporal y normalización de ventanas

--- Procesando datos por transecto ---
  Procesando Transecto_1...
  Procesando Transecto_2...

--- Procesando datos globales (por estación) ---
  Procesando T1_E1_Alicante...
  Procesando T1_E2_Elda...
  Procesando T2_E1_Elche...
  Procesando T2_E2_Elda...

Proceso completado. Revise la carpeta:
  /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/windows_partitioned
